# QTransLIFIA — Quirk → OpenQASM 2.0

Prueba los algoritmos cuánticos del repositorio. Traduce circuitos de [Quirk](https://algassert.com/quirk) a OpenQASM 2.0.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils.qutils import parse_quirk_url, quirk_col_to_qasm, quirk_to_qasm, quirk_circuit_info


In [2]:
# @title 2. Funciones auxiliares
from IPython.display import display, Markdown

ALGORITHMS = {
    "1. Shor": {
        "desc": "Algoritmo de factorización de Shor (4 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H','H','H','H'],['X'],['X','X','X','X'],[1,'•','X'],[1,'X','•'],[1,'•','X'],[1,1,'•','X'],[1,1,'X','•'],[1,1,'•','X'],['•',1,1,'X'],['X',1,1,'•'],['•',1,1,'X'],['Measure','Measure','Measure','Measure']]}",
        "offset": 0
    },
    "2. Bernstein-Vazirani": {
        "desc": "Algoritmo de Bernstein-Vazirani (4 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H','H','H'],[1,1,1,'X'],[1,1,1,'H'],['•',1,1,'X'],[1,'•',1,'X'],[1,1,'•','X'],['H','H','H'],['Measure','Measure','Measure']]}",
        "offset": 1
    },
    "3. Grover": {
        "desc": "Algoritmo de búsqueda de Grover (2 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H','H'],[1,'X'],[1,'H'],['•','X'],[1,'H'],[1,'X'],[1,'H'],[1,'Measure']]}",
        "offset": 2
    },
    "4. Deutsch-Jozsa": {
        "desc": "Algoritmo de Deutsch-Jozsa (4 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H','H','H'],['X',1,'X','X'],[1,1,1,'H'],['•',1,1,'X'],[1,'•',1,'X'],[1,1,'•','X'],['X',1,'X'],['H','H','H'],['Measure','Measure','Measure']]}",
        "offset": 0
    },
    "5. Simon": {
        "desc": "Algoritmo de Simon (6 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H','H','H'],['•',1,1,'X'],[1,'•',1,1,'X'],[1,1,'•',1,1,'X'],[1,'•',1,1,'X'],[1,'•',1,1,1,'X'],['H','H','H'],['Measure','Measure','Measure']]}",
        "offset": 1
    },
    "6. TSP": {
        "desc": "Circuito para problema del viajante (TSP, 3 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H'],[1,'X','X'],['X','•'],['X',1,'•'],['H'],['Measure','Measure','Measure']]}",
        "offset": 2
    },
    "7. Teleportation": {
        "desc": "Protocolo de teleportación cuántica (5 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[[1,'H'],[1,'•',1,1,'X'],['H'],['•','X'],['H'],['Measure','Measure'],[1,'•',1,1,'X'],['•',1,1,1,'Z']]}",
        "offset": 0
    },
    "8. Phase Estimation": {
        "desc": "Estimación de fase cuántica (4 qubits + compuertas custom)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H','H','H'],[1,1,1,'X'],['•',1,1,'Z^¼'],[1,'•',1,'Z^¼'],[1,'•',1,'Z^¼'],[1,1,'•','Z^¼'],[1,1,'•','Z^¼'],[1,1,'•','Z^¼'],[1,1,'•','Z^¼'],['Swap',1,'Swap'],['H'],['•','~16c9'],[1,'H'],['•',1,'~gf1o'],[1,'•','~16c9'],[1,1,'H'],['Measure','Measure','Measure']],'gates':[{'id':'~gf1o','name':'U(-pi/4)','matrix':'{{1,0},{0,√½-√½i}}'},{'id':'~16c9','name':'U(-pi/2)','matrix':'{{1,0},{0,-i}}'}]}",
        "offset": 1
    },
    "9. QFT": {
        "desc": "Transformada de Fourier Cuántica (3 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H','H','H'],[1,'X','X'],['H'],['Z','•'],['Z^½',1,'•'],[1,'H'],[1,'Z','•'],[1,1,'H']]}",
        "offset": 2
    },
    "10. QAOA": {
        "desc": "Quantum Approximate Optimization Algorithm (2 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['Rxft','Rxft'],[],['•','X'],['Rxft','Rxft'],[],[1,'Rxft'],[],['•','X'],['Rxft'],[],[1,'Measure'],['Measure']]}",
        "offset": 0
    },
    "11. Kickback": {
        "desc": "Phase kickback (2 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[[1,'X'],['H','H'],['•','X'],['H','H'],['Measure'],[1,'Measure']]}",
        "offset": 1
    },
    "12. Full Adder": {
        "desc": "Sumador completo (4 qubits)",
        "url": "https://algassert.com/quirk#circuit={'cols':[['H','H','H'],['•','•',1,'X'],['•','X'],[1,'•','•','X'],[1,'•','X'],['•','X'],['Measure'],[1,'Measure'],[1,1,'Measure'],[1,1,1,'Measure']]}",
        "offset": 2
    }
}

def describe_algorithms():
    md = '| # | Algoritmo | Qubits | Descripción |\n|---|---|---|---|\n'
    for k, v in ALGORITHMS.items():
        info = quirk_circuit_info(v['url'])
        nq = info['n_qubits']
        md += f'| {k.split(".")[0]} | **{k.split(". ")[1]}** | {nq} | {v["desc"]} |\n'
    display(Markdown(md))

In [3]:
# @title 3. Listar algoritmos disponibles
describe_algorithms()

| # | Algoritmo | Qubits | Descripción |
|---|---|---|---|
| 1 | **Shor** | 4 | Algoritmo de factorización de Shor (4 qubits) |
| 2 | **Bernstein-Vazirani** | 4 | Algoritmo de Bernstein-Vazirani (4 qubits) |
| 3 | **Grover** | 2 | Algoritmo de búsqueda de Grover (2 qubits) |
| 4 | **Deutsch-Jozsa** | 4 | Algoritmo de Deutsch-Jozsa (4 qubits) |
| 5 | **Simon** | 6 | Algoritmo de Simon (6 qubits) |
| 6 | **TSP** | 3 | Circuito para problema del viajante (TSP, 3 qubits) |
| 7 | **Teleportation** | 5 | Protocolo de teleportación cuántica (5 qubits) |
| 8 | **Phase Estimation** | 4 | Estimación de fase cuántica (4 qubits + compuertas custom) |
| 9 | **QFT** | 3 | Transformada de Fourier Cuántica (3 qubits) |
| 10 | **QAOA** | 2 | Quantum Approximate Optimization Algorithm (2 qubits) |
| 11 | **Kickback** | 2 | Phase kickback (2 qubits) |
| 12 | **Full Adder** | 4 | Sumador completo (4 qubits) |


In [5]:
# @title 4. Probar un algoritmo individual
ALGO = "9. QFT"  # @param ["1. Shor", "2. Bernstein-Vazirani", "3. Grover", "4. Deutsch-Jozsa", "5. Simon", "6. TSP", "7. Teleportation", "8. Phase Estimation", "9. QFT", "10. QAOA", "11. Kickback", "12. Full Adder"]

algo = ALGORITHMS[ALGO]
print(f"=== {ALGO} ===")
print(f"Descripción: {algo['desc']}")
print()
qasm = quirk_to_qasm(algo['url'], algo['offset'])
print(qasm)

=== 9. QFT ===
Descripción: Transformada de Fourier Cuántica (3 qubits)

OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[3];

h q[2];
h q[3];
h q[4];
x q[3];
x q[4];
h q[2];
cz q[3], q[2];
// controlled Z^½ not directly supported in QASM 2.0
h q[3];
cz q[4], q[3];
h q[4];


In [ ]:
# @title 5. Visualizar circuito con Qiskit
try:
    from qiskit import QuantumCircuit
    from qiskit.visualization import circuit_drawer

    qc = QuantumCircuit.from_qasm_str(qasm)
    display(circuit_drawer(qc, output='mpl', scale=0.7))
except ImportError:
    print("qiskit no está instalado. Corré la celda 1 primero.")
except Exception as e:
    print(f"Error al dibujar: {e}")

In [ ]:
# @title 6. Simular y ver resultados
try:
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator

    qc = QuantumCircuit.from_qasm_str(qasm)
    sim = AerSimulator()

    # Contar cuántas mediciones hay
    n_meas = sum(1 for line in qasm.split('\n') if 'measure' in line)
    shots = 1024 if n_meas > 0 else 1

    if n_meas > 0:
        result = sim.run(qc, shots=shots).result()
        counts = result.get_counts()
        print(f"Resultados ({shots} shots):")
        from qiskit.visualization import plot_histogram
        display(plot_histogram(counts))
    else:
        # Sin mediciones: mostrar matriz unitaria
        from qiskit.quantum_info import Operator
        unitary = Operator(qc).data
        import numpy as np
        np.set_printoptions(precision=3, suppress=True)
        print("Matriz unitaria:")
        print(unitary)
except ImportError:
    print("qiskit o qiskit-aer no están instalados.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# @title 7. Probar TODOS los algoritmos
results = {}
for name, algo in ALGORITHMS.items():
    try:
        qasm = quirk_to_qasm(algo['url'], algo['offset'])
        results[name] = {'status': 'OK', 'qasm': qasm}
    except Exception as e:
        results[name] = {'status': f'ERROR: {e}', 'qasm': ''}

md = '| Algoritmo | Status | Líneas QASM |\n|---|---|---|\n'
for name, r in results.items():
    lines = len(r['qasm'].split('\n')) if r['qasm'] else 0
    md += f'| {name} | {r["status"]} | {lines} |\n'
display(Markdown(md))

In [ ]:
# @title 8. Probar URL personalizada
CUSTOM_URL = "https://algassert.com/quirk#circuit={'cols':[['H'],['•','X'],['Measure','Measure']]}"  # @param {type:"string"}
CUSTOM_OFFSET = 0  # @param {type:"integer"}

try:
    qasm = quirk_to_qasm(CUSTOM_URL, CUSTOM_OFFSET)
    print(qasm)
except Exception as e:
    print(f"Error: {e}")